# 5 · Retrieval (all 4 configurations)

Runs the LangChain `HybridLegalRetriever` for each configuration over the 15 benchmark queries and saves the retrieved contexts to `results/contexts/`.

Unchunked configs use the original question; rechunked configs use the expanded query and add BM25 + cross-encoder re-ranking. **Machine 2** (needs local models).

In [ ]:
import json
import pandas as pd
from config import config
from indian_marriage_legal_recommender.retrieval import get_retriever

df = pd.read_csv(config.BENCHMARK_EXPANDED_CSV)
config.CONTEXTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
for key, prof in config.PROFILES.items():
    retriever = get_retriever(key, top_k_final=config.TOP_K_FINAL)
    out = []
    for i, row in df.iterrows():
        query = row['Expanded_Query'] if prof.use_expanded_query else row['Question']
        docs = retriever.invoke(query)
        out.append({'id': int(i), 'query': row['Question'],
                    'contexts': [d.page_content for d in docs]})
    path = config.CONTEXTS_DIR / f'{key}_contexts.json'
    path.write_text(json.dumps(out, ensure_ascii=False, indent=2))
    print('Saved', path)